# 04 — RQ1: Random Window Split vs. File-based Split + LOLO

Notebook này thực hiện RQ1 về tính trung thực của giao thức kiểm định. So sánh Random Window Split với File-based Split + LOLO trên cùng bộ 32 đặc trưng và cùng mô hình RF baseline.

**Random Window Split chỉ là đối chứng leakage; kết quả chính thức của đề tài dựa trên File-based Split + LOLO.**


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from common import training


In [ ]:
OUTPUT_DIR = Path('./outputs')
TABLES_DIR = OUTPUT_DIR / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

feature_df = pd.read_parquet('../giai_doan_1_tien_xu_ly/outputs/tables/features_mlp.parquet')

time_cols = [c for c in feature_df.columns if c.startswith('time_')]
order_cols = [c for c in feature_df.columns if c.startswith('order_')]
envelope_cols = [c for c in feature_df.columns if c.startswith('envelope_')]
feature_cols = time_cols + order_cols + envelope_cols

assert len(time_cols) == 11
assert len(order_cols) == 12
assert len(envelope_cols) == 9
assert len(feature_cols) == 32

print(f'32 features: Time={len(time_cols)}, Order={len(order_cols)}, Envelope={len(envelope_cols)}')


## 1. File-based Split + LOLO — giao thức chuẩn

Mỗi fold giữ một mức tải làm Test. Ba mức tải còn lại được chia theo file gốc; không trộn các window gần trùng từ cùng file vào cả Train và Test.


In [ ]:
def rf_factory():
    return RandomForestClassifier(n_estimators=100, random_state=42)

per_fold_lolo, summary_lolo = training.run_lolo_evaluation(
    feature_df,
    feature_cols=feature_cols,
    estimator_factory=rf_factory,
    label_col='label',
    load_col='load_hp',
    val_ratio=0.2,
    seed=42,
    loads=(0,1,2,3),
    use_val_for_fit=True
)

per_fold_lolo.to_csv(TABLES_DIR / 'rq1_filebased_lolo_per_fold.csv', index=False)
pd.DataFrame([summary_lolo]).to_csv(TABLES_DIR / 'rq1_filebased_lolo_summary.csv', index=False)
print(summary_lolo)


## 2. Random Window Split — đối chứng leakage

Toàn bộ feature rows được chia ngẫu nhiên. Do sliding windows từ cùng file có thể xuất hiện ở cả Train và Test, đây không phải giao thức đánh giá generalization hợp lệ.


In [ ]:
X = feature_df[feature_cols].values
y = feature_df['label'].values
le = LabelEncoder()
y_enc = le.fit_transform(y)

random_rows = []
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_enc,
        test_size=0.25,
        random_state=seed,
        stratify=y_enc
    )
    rf = rf_factory()
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    random_rows.append({
        'seed': seed,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_macro': f1_score(y_test, y_pred, average='macro')
    })

random_df = pd.DataFrame(random_rows)
random_df.to_csv(TABLES_DIR / 'rq1_random_window_per_seed.csv', index=False)
random_summary = pd.DataFrame([{
    'method': 'Random Window Split',
    'accuracy_mean': random_df['accuracy'].mean(),
    'accuracy_std': random_df['accuracy'].std(ddof=0),
    'f1_mean': random_df['f1_macro'].mean(),
    'f1_std': random_df['f1_macro'].std(ddof=0),
}])
display(random_summary)


In [ ]:
lolo_summary = pd.DataFrame([{
    'method': 'File-based Split + LOLO',
    'accuracy_mean': summary_lolo['accuracy_mean'],
    'accuracy_std': summary_lolo['accuracy_std'],
    'f1_mean': summary_lolo['f1_macro_mean'],
    'f1_std': summary_lolo['f1_macro_std'],
}])

comparison = pd.concat([lolo_summary, random_summary], ignore_index=True)
comparison['delta_accuracy_vs_lolo'] = comparison['accuracy_mean'] - lolo_summary.loc[0, 'accuracy_mean']
comparison.to_csv(TABLES_DIR / 'rq1_random_vs_filebased_lolo.csv', index=False)
display(comparison)


## 3. Metric báo cáo cho RQ1

- Accuracy và F1-macro: báo cáo **mean ± std**.
- `Delta Accuracy = Accuracy(Random Window Split) − Accuracy(File-based + LOLO)`.
- Random Window Split chỉ dùng để minh họa mức độ lạc quan giả do leakage, không dùng làm kết quả deploy/generalization.
